In [ ]:
!git clone https://github.com/huggingface/nanoVLM.git
%cd nanoVLM

!pip install -q datasets huggingface-hub transformers pillow tqdm thop

# Change:

- train_iter_idx += 1 - шли попорядку по массиву тренировки
- делаем проход рандомным: idx = random.randrange(len(ds_train))

- Мощнее голова: MLP + dropout

In [ ]:
%%writefile /kaggle/working/nanoVLM/train_aokvqa_mcq_ddp.py
import argparse, math, random, os, contextlib
from pathlib import Path
from typing import List, Dict, Any

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

from models.vision_language_model import VisionLanguageModel
from data.processors import get_tokenizer, get_image_processor

# -----------------------------
# MCQ head + forward_mcq_logits
# -----------------------------

class MCQHead(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 4),   # 4 варианта ответа
        )

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        last = hidden[:, -1, :]
        return self.net(last)

def forward_mcq_logits(self,
                       input_ids: torch.Tensor,
                       images: torch.Tensor,
                       attention_mask: torch.Tensor | None = None) -> torch.Tensor:
    token_embd = self.decoder.token_embedding(input_ids)

    images_tensor = self._process_images(images, input_ids.device)
    if images_tensor is not None:
        image_embd = self.vision_encoder(images_tensor)
        image_embd = self.MP(image_embd)
        token_embd = self._replace_img_tokens_with_embd(input_ids, token_embd, image_embd)

    hidden, _ = self.decoder(
        token_embd,
        attention_mask=attention_mask,
        kv_cache=None,
        start_pos=0
    )
    logits = self.mcq_head(hidden)
    return logits

def mcq_forward(model,
                input_ids: torch.Tensor,
                images: torch.Tensor,
                distributed: bool,
                attention_mask: torch.Tensor | None = None) -> torch.Tensor:
    if distributed and isinstance(model, nn.parallel.DistributedDataParallel):
        return model.module.forward_mcq_logits(input_ids, images, attention_mask=attention_mask)
    else:
        return model.forward_mcq_logits(input_ids, images, attention_mask=attention_mask)

# подвешиваем новый forward к классу
VisionLanguageModel.forward_mcq_logits = forward_mcq_logits

LETTER_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3}
IDX_TO_LETTER = "ABCD"

def seed_everything(seed=1234):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def init_distributed():
    """
    Инициализация DDP, если скрипт запущен через torchrun.
    Возвращает (distributed, rank, world_size, local_rank).
    """
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        rank = int(os.environ["RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        local_rank = int(os.environ.get("LOCAL_RANK", 0))
        dist.init_process_group(backend="nccl")
        torch.cuda.set_device(local_rank)
        return True, rank, world_size, local_rank
    else:
        return False, 0, 1, 0

def is_main_process(rank: int) -> bool:
    return rank == 0

def build_mc_prompt(question: str, choices: List[str]) -> str:
    return (
        "You are a visual question-answering assistant.\n"
        "Read the question and choose exactly one correct option.\n\n"
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n"
        "Answer (A, B, C, or D):"
    )

class AOKVQAMCQRawDataset(Dataset):
    def __init__(self, split="train"):
        self.ds = load_dataset("HuggingFaceM4/A-OKVQA", split=split)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.ds[int(idx)]
        return {
            "image": ex["image"],
            "question": ex["question"],
            "choices": list(ex["choices"]),
            "gt_idx": int(ex["correct_choice_idx"]),
            "qid": str(ex["question_id"])
        }

class AOKVQAMCQPreprocDataset(Dataset):
    def __init__(self,
                 split: str,
                 tokenizer,
                 imgproc,
                 mp_image_token_length: int):
        self.ds = load_dataset("HuggingFaceM4/A-OKVQA", split=split)
        self.tokenizer = tokenizer
        self.imgproc = imgproc
        self.mp_image_token_length = mp_image_token_length

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.ds[int(idx)]
        img: Image.Image = ex["image"].convert("RGB")
        q = ex["question"]
        choices = list(ex["choices"])
        gt_idx = int(ex["correct_choice_idx"])

        proc_img, _ = self.imgproc(img)          # [n_imgs, 3, H, W]
        n_imgs = proc_img.shape[0]
        num_img_tokens = self.mp_image_token_length * n_imgs
        image_tokens = self.tokenizer.image_token * num_img_tokens

        prompt = build_mc_prompt(q, choices)
        messages = [{"role": "user", "content": image_tokens + prompt}]
        ids = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True
        )
        input_ids = torch.tensor(ids, dtype=torch.long)   # [T]

        return {
            "input_ids": input_ids,
            "images": proc_img,       # [n_imgs, 3, H, W]
            "label": gt_idx,
        }

# -----------------------------
# Настройка trainable модулей (partial unfreeze)
# -----------------------------

def configure_trainable_modules(model, args, rank):
    """
    Настраиваем, какие части модели обучаем.
    """
    if args.freeze_vision and hasattr(model, "vision_encoder"):
        for p in model.vision_encoder.parameters():
            p.requires_grad = False
        if is_main_process(rank):
            print("[freeze] vision_encoder")

    # MP
    if args.freeze_proj and hasattr(model, "MP"):
        for p in model.MP.parameters():
            p.requires_grad = False
        if is_main_process(rank):
            print("[freeze] modality projector (MP)")
    else:
        if is_main_process(rank):
            print("[unfreeze] modality projector (MP)")

    # decoder
    dec = model.decoder

    if hasattr(dec, "layers"):
        layers = dec.layers
    elif hasattr(dec, "model") and hasattr(dec.model, "layers"):
        layers = dec.model.layers
    else:
        if is_main_process(rank):
            print("[warn] cannot find decoder layers (no .layers or .model.layers). "
                  "Decoder will stay as is.")
        return

    num_layers = len(layers)
    k = max(0, int(args.unfreeze_decoder_layers))

    if k == 0:
        for layer in layers:
            for p in layer.parameters():
                p.requires_grad = False
        if is_main_process(rank):
            print(f"[freeze] all {num_layers} decoder layers")
    else:
        cutoff = num_layers - k
        for i, layer in enumerate(layers):
            train_this = (i >= cutoff)
            for p in layer.parameters():
                p.requires_grad = train_this
        if is_main_process(rank):
            print(f"[freeze] first {cutoff} decoder layers, "
                  f"[unfreeze] last {k} decoder layers")

# -----------------------------
# TRAIN с DDP
# -----------------------------

def train(args):
    seed_everything(args.seed)

    distributed, rank, world_size, local_rank = init_distributed()
    device = torch.device("cuda", local_rank) if torch.cuda.is_available() else torch.device("cpu")

    if is_main_process(rank):
        print(f"Distributed: {distributed} | world_size={world_size} | rank={rank}")
        print(f"Device: {device}")
        print(f"Loading model: {args.model_id}")

    model = VisionLanguageModel.from_pretrained(args.model_id).to(device)

    # навешиваем MLP MCQ-голову
    hidden_dim = model.decoder.token_embedding.embedding_dim
    model.mcq_head = MCQHead(hidden_dim, dropout=args.head_dropout).to(device)

    # cfg
    mp_image_token_length = model.cfg.mp_image_token_length
    max_img_size = model.cfg.max_img_size
    vit_img_size = model.cfg.vit_img_size

    configure_trainable_modules(model, args, rank)

    if distributed:
        model = nn.parallel.DistributedDataParallel(
            model,
            device_ids=[local_rank],
            output_device=local_rank,
            find_unused_parameters=False,
        )

    cfg = model.module.cfg if distributed else model.cfg
    tokenizer = get_tokenizer(
        cfg.lm_tokenizer,
        cfg.vlm_extra_tokens,
        cfg.lm_chat_template,
    )
    imgproc = get_image_processor(max_img_size, vit_img_size)

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id

    train_dataset = AOKVQAMCQPreprocDataset(
        split="train",
        tokenizer=tokenizer,
        imgproc=imgproc,
        mp_image_token_length=mp_image_token_length,
    )
    val_dataset_raw = AOKVQAMCQRawDataset(split="validation")

    if is_main_process(rank):
        print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset_raw)}")

    if distributed:
        train_sampler = DistributedSampler(
            train_dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=True,
        )
    else:
        train_sampler = None

    def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_size = len(batch)
        max_len = max(ex["input_ids"].size(0) for ex in batch)

        input_ids = torch.full((batch_size, max_len),
                               fill_value=pad_id,
                               dtype=torch.long)
        attention_mask = torch.zeros((batch_size, max_len),
                                     dtype=torch.long)

        images_list = []
        labels = torch.empty(batch_size, dtype=torch.long)

        for i, ex in enumerate(batch):
            ids = ex["input_ids"]
            L = ids.size(0)
            input_ids[i, :L] = ids
            attention_mask[i, :L] = 1

            img = ex["images"]      # [n_imgs_i, 3, H, W]
            images_list.append(img)

            labels[i] = int(ex["label"])

        # Глобальный тензор картинок: [sum_i n_imgs_i, 3, H, W]
        images = torch.cat(images_list, dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "images": images,
            "labels": labels,
        }

    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        sampler=train_sampler,
        shuffle=(train_sampler is None),
        num_workers=args.num_workers,
        pin_memory=True,
        collate_fn=collate_fn,
        drop_last=False,
    )

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if is_main_process(rank):
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params_cnt = sum(p.numel() for p in trainable_params)
        print(f"Total params: {total_params/1e6:.2f}M | Trainable: {trainable_params_cnt/1e6:.2f}M")

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=args.lr,
        betas=(0.9, 0.95),
        weight_decay=args.weight_decay,
    )

    updates_per_epoch = math.ceil(len(train_loader) / max(1, args.grad_accum))
    total_steps = updates_per_epoch * args.epochs
    warmup_steps = int(args.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, (total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
    global_step, best_val = 0, -1.0

    @torch.no_grad()
    def evaluate_mcq(split_ds) -> float:
        assert is_main_process(rank)
        model.eval()
        correct = 0
        total = len(split_ds)
        pbar = tqdm(split_ds, total=total, desc="Eval-MCQ", dynamic_ncols=True)
        for ex in pbar:
            img: Image.Image = ex["image"].convert("RGB")
            q, choices = ex["question"], list(ex["choices"])
            gt_idx = int(ex["gt_idx"])

            proc_img, _ = imgproc(img)
            n_imgs = proc_img.shape[0]
            num_img_tokens = mp_image_token_length * n_imgs
            image_tokens = tokenizer.image_token * num_img_tokens

            prompt = build_mc_prompt(q, choices)
            messages = [{"role": "user", "content": image_tokens + prompt}]
            ids = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True
            )
            input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
            attention_mask = torch.ones_like(input_ids, device=device)

            logits = mcq_forward(
                model,
                input_ids,
                proc_img.to(device),
                distributed,
                attention_mask=attention_mask,
            )
            pred_idx = int(logits.argmax(dim=-1).item())
            correct += int(pred_idx == gt_idx)

        model.train()
        return 100.0 * correct / total

    optimizer.zero_grad(set_to_none=True)

    for epoch in range(args.epochs):
        if distributed and isinstance(train_sampler, DistributedSampler):
            train_sampler.set_epoch(epoch)

        if is_main_process(rank):
            epoch_pbar = tqdm(
                enumerate(train_loader),
                total=len(train_loader),
                desc=f"Epoch {epoch+1}/{args.epochs}",
                dynamic_ncols=True,
            )
        else:
            epoch_pbar = enumerate(train_loader)

        for step_in_epoch, batch in epoch_pbar:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            images = batch["images"].to(device, non_blocking=True)   # [sum_n_imgs, 3, H, W]
            labels = batch["labels"].to(device, non_blocking=True)

            micro_step = step_in_epoch % args.grad_accum
            sync_grad = (micro_step == args.grad_accum - 1)

            if distributed and isinstance(model, nn.parallel.DistributedDataParallel) and not sync_grad:
                ctx = model.no_sync()
            else:
                ctx = contextlib.nullcontext()

            with ctx:
                logits = mcq_forward(
                    model,
                    input_ids,
                    images,
                    distributed,
                    attention_mask=attention_mask,
                )
                loss = loss_fn(logits, labels)
                loss = loss / args.grad_accum
                loss.backward()

            if sync_grad:
                clip_grad_norm_(trainable_params, args.grad_clip)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if is_main_process(rank):
                    epoch_pbar.set_postfix(
                        loss=f"{loss.item()*args.grad_accum:.4f}",
                        lr=f"{scheduler.get_last_lr()[0]:.2e}",
                    )

                if is_main_process(rank) and (global_step % args.eval_every == 0):
                    val_acc = evaluate_mcq(val_dataset_raw)
                    print(f"\n[Eval] step={global_step} | val MCQ acc = {val_acc:.2f}%")
                    if val_acc > best_val:
                        best_val = val_acc
                        ckpt_path = Path(args.output_dir)
                        ckpt_path.mkdir(parents=True, exist_ok=True)
                        ckpt_path = ckpt_path / f"best_mcq_{val_acc:.2f}.pt"
                        print(f"[Checkpoint] Saving to {ckpt_path}")
                        torch.save({
                            "model_state_dict": model.module.state_dict() if distributed else model.state_dict(),
                            "val_acc": val_acc,
                            "step": global_step
                        }, ckpt_path)

    if is_main_process(rank):
        final_acc = evaluate_mcq(val_dataset_raw)
        print(f"\n[Final] val MCQ acc = {final_acc:.2f}%  (best={best_val:.2f}%)")

        full_out = Path(args.output_dir) / "mcq_finetuned"
        full_out.mkdir(parents=True, exist_ok=True)
        state_dict = model.module.state_dict() if distributed else model.state_dict()
        torch.save(state_dict, full_out / "pytorch_model.bin")
        print(f"Saved state_dict to: {full_out}")

    if distributed:
        dist.destroy_process_group()

def cli():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_id", default="lusxvr/nanoVLM")
    ap.add_argument("--output_dir", default="checkpoints_mcq")
    ap.add_argument("--epochs", type=int, default=1)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--weight_decay", type=float, default=0.01)
    ap.add_argument("--warmup_ratio", type=float, default=0.10)

    ap.add_argument("--batch_size", type=int, default=2)
    ap.add_argument("--grad_accum", type=int, default=4)
    ap.add_argument("--grad_clip", type=float, default=1.0)
    ap.add_argument("--eval_every", type=int, default=500)

    ap.add_argument("--num_workers", type=int, default=4)

    ap.add_argument("--label_smoothing", type=float, default=0.1)
    ap.add_argument("--head_dropout", type=float, default=0.1)
    ap.add_argument("--unfreeze_decoder_layers", type=int, default=4)

    ap.add_argument("--freeze_vision", action="store_true", default=True)
    ap.add_argument("--freeze_proj", action="store_true", default=False)
    ap.add_argument("--seed", type=int, default=1234)
    args = ap.parse_args()
    train(args)

if __name__ == "__main__":
    cli()


# Launch

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/nanoVLM/train_aokvqa_mcq_ddp.py \
  --model_id lusxvr/nanoVLM \
  --output_dir checkpoints_mcq_ddp_batched \
  --epochs 1 \
  --lr 1e-4 \
  --batch_size 2 \
  --grad_accum 4 \
  --label_smoothing 0.1 \
  --head_dropout 0.1 \
  --unfreeze_decoder_layers 4 \
  --freeze_vision \
  --num_workers 4



# Save

In [ ]:
import os, json

os.makedirs("/root/.config/kaggle", exist_ok=True)
creds = {
    "username": os.environ["KAGGLE_USERNAME"],
    "key": os.environ["KAGGLE_KEY"],
}
with open("/root/.config/kaggle/kaggle.json", "w") as f:
    json.dump(creds, f)
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)

print("Kaggle credentials configured from environment.")


In [ ]:
import os
import json
import subprocess
from pathlib import Path

# ===========================
# Конфигурация
# ===========================
username = os.environ["KAGGLE_USERNAME"]     # твой Kaggle username
dataset_name = "nanovlm-mcq-par-upd-arch-exp4-5"   # НОВОЕ имя датасета для каждой загрузки

model_dir = "checkpoints_mcq_ddp_mlp_ls"
publish_dir = f"/kaggle/working/publish_{dataset_name}"

# ===========================
# Подготовка директории
# ===========================
os.makedirs(publish_dir, exist_ok=True)

# копируем файлы модели
os.system(f"cp -r {model_dir}/* {publish_dir}/")

# ===========================
# Создаём dataset-metadata.json
# ===========================
metadata = {
    "title": f"nanoVLM MCQ Model {dataset_name}",
    "id": f"{username}/{dataset_name}",
    "licenses": [{"name": "Apache-2.0"}]
}

with open(Path(publish_dir) / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# ===========================
# Публикация ДАТАСЕТА (НОВОГО)
# ===========================
print("Создаём новый датасет…")

proc = subprocess.run(
    ["kaggle", "datasets", "create", "-p", publish_dir, "--dir-mode", "zip"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print(proc.stdout)


# Inference

In [ ]:
%%writefile /kaggle/working/nanoVLM/infer_aokvqa_mcq_ddp.py
import argparse, os, math, time, random
from pathlib import Path
from typing import List, Dict, Any

import torch
import torch.nn as nn
import torch.distributed as dist
import zipfile
import matplotlib
matplotlib.use("Agg")  # рисуем в память, без GUI

import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, load_dataset_builder
from PIL import Image
from tqdm import tqdm

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score,
)

from thop import profile

from models.vision_language_model import VisionLanguageModel
from data.processors import get_tokenizer, get_image_processor

# =========================
# MCQ head + forward_mcq_logits
# =========================

class MCQHead(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.out = nn.Linear(hidden_dim, 4)

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        last = hidden[:, -1, :]
        return self.out(last)

def forward_mcq_logits(self, input_ids: torch.Tensor, images: torch.Tensor, attention_mask=None) -> torch.Tensor:
    token_embd = self.decoder.token_embedding(input_ids)

    images_tensor = self._process_images(images, input_ids.device)
    if images_tensor is not None:
        image_embd = self.vision_encoder(images_tensor)
        image_embd = self.MP(image_embd)
        token_embd = self._replace_img_tokens_with_embd(input_ids, token_embd, image_embd)

    hidden, _ = self.decoder(
        token_embd,
        attention_mask=attention_mask,
        kv_cache=None,
        start_pos=0
    )
    logits = self.mcq_head(hidden)  # [B, 4]
    return logits

VisionLanguageModel.forward_mcq_logits = forward_mcq_logits

LETTERS = "ABCD"

# =========================
# DDP utils
# =========================

def seed_everything(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def init_distributed():
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        rank = int(os.environ["RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        local_rank = int(os.environ.get("LOCAL_RANK", 0))
        dist.init_process_group(backend="nccl")
        torch.cuda.set_device(local_rank)
        return True, rank, world_size, local_rank
    else:
        return False, 0, 1, 0

def is_main_process(rank: int) -> bool:
    return rank == 0

def mcq_forward(model, input_ids, images, distributed: bool):
    if distributed and isinstance(model, nn.parallel.DistributedDataParallel):
        return model.module.forward_mcq_logits(input_ids, images)
    else:
        return model.forward_mcq_logits(input_ids, images)

# =========================
# Dataset
# =========================

class AOKVQAMCQDataset(torch.utils.data.Dataset):
    def __init__(self, split="validation"):
        self.ds = load_dataset("HuggingFaceM4/A-OKVQA", split=split)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.ds[int(idx)]
        return {
            "image": ex["image"],
            "question": ex["question"],
            "choices": list(ex["choices"]),
            "gt_idx": int(ex["correct_choice_idx"]),
            "qid": str(ex["question_id"]),
        }

def build_mc_prompt(question: str, choices: List[str]) -> str:
    return (
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n"
        "Answer:"
    )

# =========================
# All-gather helper for 1D tensors (variable length)
# =========================

def gather_1d_tensor(t: torch.Tensor, world_size: int, rank: int):
    """
    t: 1D tensor on each rank (can be different length).
    Возвращает на rank 0 конкатенированный tensor, на остальных None.
    """
    device = t.device
    length = torch.tensor([t.numel()], device=device, dtype=torch.long)
    lengths = [torch.zeros_like(length) for _ in range(world_size)]
    dist.all_gather(lengths, length)
    lengths = [l.item() for l in lengths]
    max_len = max(lengths)

    if t.numel() < max_len:
        pad = torch.zeros(max_len - t.numel(), dtype=t.dtype, device=device)
        t_padded = torch.cat([t, pad], dim=0)
    else:
        t_padded = t

    gather_list = [torch.zeros_like(t_padded) for _ in range(world_size)]
    dist.all_gather(gather_list, t_padded)

    if rank == 0:
        parts = []
        for i, g in enumerate(gather_list):
            parts.append(g[:lengths[i]].cpu())
        return torch.cat(parts, dim=0)
    else:
        return None

# =========================
# Main inference routine
# =========================
def run_inference(args):
    seed_everything(args.seed)

    metrics_dir = Path(args.metrics_dir)
    if is_main_process(0):  # ранг ещё не инициализирован, но это ок
        metrics_dir.mkdir(parents=True, exist_ok=True)

    distributed, rank, world_size, local_rank = init_distributed()
    device = torch.device("cuda", local_rank) if torch.cuda.is_available() else torch.device("cpu")
    metrics_dir = Path(args.metrics_dir)
    if is_main_process(rank):
        metrics_dir.mkdir(parents=True, exist_ok=True)
        print(f"Distributed: {distributed} | world_size={world_size} | rank={rank}")
        print("Device:", device)
        print("Loading model:", args.model_id)

    # базовая модель
    model = VisionLanguageModel.from_pretrained(args.model_id).to(device)

    # навешиваем MCQ-голову
    hidden_dim = model.decoder.token_embedding.embedding_dim
    model.mcq_head = MCQHead(hidden_dim).to(device)

    # если есть чекпоинт — подгружаем веса
    if args.ckpt_path is not None and args.ckpt_path != "":
        if is_main_process(rank):
            print("Loading checkpoint:", args.ckpt_path)
        ckpt = torch.load(args.ckpt_path, map_location="cpu")
        state_dict = ckpt.get("model_state_dict", ckpt)
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        if is_main_process(rank):
            print("Loaded state_dict. Missing keys:", missing)
            print("Unexpected keys:", unexpected)

    if distributed:
        model = nn.parallel.DistributedDataParallel(
            model,
            device_ids=[local_rank],
            output_device=local_rank,
            find_unused_parameters=False,
        )

    cfg = model.module.cfg if distributed else model.cfg
    tokenizer = get_tokenizer(cfg.lm_tokenizer, cfg.vlm_extra_tokens, cfg.lm_chat_template)
    imgproc = get_image_processor(cfg.max_img_size, cfg.vit_img_size)
    mp_image_token_length = cfg.mp_image_token_length

    # инфо о датасете
    if is_main_process(rank):
        builder = load_dataset_builder("HuggingFaceM4/A-OKVQA")
        info = builder.info
        download_gb = info.download_size / (1024**3)
        dataset_gb  = info.dataset_size  / (1024**3)
        print(f"Плановый объём скачивания (raw): {download_gb:.2f} GB")
        print(f"Размер закэшированного датасета: {dataset_gb:.2f} GB")

    ds_val = AOKVQAMCQDataset(split="validation")
    total_ds = len(ds_val)

    if args.max_examples > 0:
        total_ds = min(total_ds, args.max_examples)

    if is_main_process(rank):
        print("A-OKVQA val size:", len(ds_val))
        print("Using first", total_ds, "examples for eval")

    # FLOPs на одном примере (только rank 0)
    flops_per_forward = None
    params = None
    if is_main_process(rank):
        class MCQWrapper(nn.Module):
            def __init__(self, vlm):
                super().__init__()
                self.vlm = vlm

            def forward(self, input_ids, images):
                return self.vlm.forward_mcq_logits(input_ids, images)

        mcq_wrapper = MCQWrapper(model.module if distributed else model).to(device).eval()

        with torch.no_grad():
            ex0 = ds_val[0]
            img0: Image.Image = ex0["image"].convert("RGB")
            q0 = ex0["question"]
            choices0 = list(ex0["choices"])

            proc_img0, _ = imgproc(img0)
            n_imgs0 = proc_img0.shape[0]
            num_img_tokens0 = mp_image_token_length * n_imgs0
            image_tokens0 = tokenizer.image_token * num_img_tokens0

            prompt0 = build_mc_prompt(q0, choices0)
            messages0 = [{"role": "user", "content": image_tokens0 + prompt0}]
            ids0 = tokenizer.apply_chat_template(
                messages0,
                tokenize=True,
                add_generation_prompt=True
            )
            input_ids0 = torch.tensor(ids0, dtype=torch.long, device=device).unsqueeze(0)

            flops_per_forward, params = profile(
                mcq_wrapper,
                inputs=(input_ids0, proc_img0.to(device)),
                verbose=False
            )

        print(f"FLOPs per forward_mcq_logits: {flops_per_forward:.3e}")
        print(f"Params in MCQ pipeline:       {params:.3e}")

    # если нужно, можно раскидать flops по всем процессам, но для метрик достаточно rank 0
    if distributed:
        # broadcast flops/perams чтобы не держать None на других
        flops_tensor = torch.zeros(2, dtype=torch.float64, device=device)
        if is_main_process(rank):
            flops_tensor[0] = float(flops_per_forward if flops_per_forward is not None else 0.0)
            flops_tensor[1] = float(params if params is not None else 0.0)
        dist.broadcast(flops_tensor, src=0)
        flops_per_forward = float(flops_tensor[0].item())
        params = float(flops_tensor[1].item())

    # раздаём индексы по rank'ам
    indices = list(range(total_ds))
    per_rank = (total_ds + world_size - 1) // world_size
    start = rank * per_rank
    end = min(start + per_rank, total_ds)
    local_indices = indices[start:end]

    if is_main_process(rank):
        print(f"indices per rank ~ {per_rank}, rank0 range = [0, {min(per_rank, total_ds)})")

    model.eval()

    correct_local = 0
    num_forwards_local = 0
    y_true_pairs_local = []
    y_score_pairs_local = []

    lat_e2e_local = []
    lat_pre_local = []
    lat_det_local = []

    err_pred_counts_local = [0, 0, 0, 0]  # по предсказанным вариантам среди ошибок

    # логирование примеров
    try:
        from IPython.display import display
    except ImportError:
        display = None

    logged = 0
    n_show_first = args.n_show_first

    if is_main_process(rank):
        it = tqdm(local_indices, desc="Inference (rank 0)", dynamic_ncols=True)
    else:
        it = local_indices
    logged = 0
    n_show_first = args.n_show_first
    seen_local = 0

    for ex_idx in it:
        t0 = time.perf_counter()

        ex = ds_val[ex_idx]
        img: Image.Image = ex["image"].convert("RGB")
        q = ex["question"]
        choices = list(ex["choices"])
        gt_idx = int(ex["gt_idx"])

        # препроцессинг
        t_pre0 = time.perf_counter()
        proc_img, _ = imgproc(img)
        n_imgs = proc_img.shape[0]
        num_img_tokens = mp_image_token_length * n_imgs
        image_tokens = tokenizer.image_token * num_img_tokens

        prompt = build_mc_prompt(q, choices)
        messages = [{"role": "user", "content": image_tokens + prompt}]
        ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True
        )
        input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
        t_pre1 = time.perf_counter()

        # forward
        t_det0 = time.perf_counter()
        logits = mcq_forward(model, input_ids, proc_img.to(device), distributed).squeeze(0)
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        t_det1 = time.perf_counter()

        num_forwards_local += 1

        pred_idx = int(np.argmax(probs))
        is_correct = (pred_idx == gt_idx)
        seen_local += 1
        if is_correct:
            correct_local += 1
        else:
            err_pred_counts_local[pred_idx] += 1


        # pairwise статистика
        for j in range(len(choices)):
            y_true_pairs_local.append(1 if j == gt_idx else 0)
            y_score_pairs_local.append(float(probs[j]))

        # логирование только первых n_show_first примеров на rank 0
        if is_main_process(rank) and logged < n_show_first:
            logged += 1
            print(f"\n=== Example {logged} (global idx {ex_idx}) ===")
            if display is not None:
                display(img)
            print("Question:", q)
            print("Choices and probs:")
            for i, (p, ch) in enumerate(zip(probs, choices)):
                letter = LETTERS[i]
                marker_gt = "<-- GT" if i == gt_idx else ""
                marker_pred = "<-- PRED" if i == pred_idx else ""
                print(f"  {letter}) {p:.4f} | {ch} {marker_gt} {marker_pred}")
            print(f"Final predicted: {LETTERS[pred_idx]}  (correct={is_correct})")

        # latency
        t1 = time.perf_counter()
        lat_pre_local.append(t_pre1 - t_pre0)
        lat_det_local.append(t_det1 - t_det0)
        lat_e2e_local.append(t1 - t0)
        # обновление прогресс-бара глобальными статистиками
        if distributed:
            stats = torch.tensor(
                [correct_local, seen_local],
                dtype=torch.long,
                device=device,
            )
            dist.all_reduce(stats, op=dist.ReduceOp.SUM)
            correct_now, seen_now = stats.tolist()
        else:
            correct_now, seen_now = correct_local, seen_local

        if is_main_process(rank) and isinstance(it, tqdm):
            wrong_now = seen_now - correct_now
            acc_now = 100.0 * correct_now / max(1, seen_now)
            it.set_postfix(
                correct=correct_now,
                wrong=wrong_now,
                acc=f"{acc_now:.2f}%"
            )

    # =========================
    # Сбор метрик с ранков
    # =========================

    local_counts = torch.tensor(
        [len(local_indices), correct_local, num_forwards_local],
        dtype=torch.long, device=device
    )

    if distributed:
        dist.reduce(local_counts, dst=0, op=dist.ReduceOp.SUM)

    if is_main_process(rank):
        total_examples, correct_total, num_forwards_total = local_counts.tolist()
    else:
        total_examples = correct_total = num_forwards_total = None

    # ошибки по вариантам
    err_counts_t = torch.tensor(err_pred_counts_local, dtype=torch.long, device=device)
    if distributed:
        dist.reduce(err_counts_t, dst=0, op=dist.ReduceOp.SUM)
    if is_main_process(rank):
        err_counts = err_counts_t.cpu().tolist()
    else:
        err_counts = None

    # времена
    lat_e2e_t = torch.tensor(lat_e2e_local, dtype=torch.float64, device=device)
    lat_pre_t = torch.tensor(lat_pre_local, dtype=torch.float64, device=device)
    lat_det_t = torch.tensor(lat_det_local, dtype=torch.float64, device=device)

    if distributed:
        lat_e2e_all = gather_1d_tensor(lat_e2e_t, world_size, rank)
        lat_pre_all = gather_1d_tensor(lat_pre_t, world_size, rank)
        lat_det_all = gather_1d_tensor(lat_det_t, world_size, rank)
    else:
        lat_e2e_all = lat_e2e_t.cpu()
        lat_pre_all = lat_pre_t.cpu()
        lat_det_all = lat_det_t.cpu()

    # pairwise y_true / y_score
    y_true_t = torch.tensor(y_true_pairs_local, dtype=torch.int64, device=device)
    y_score_t = torch.tensor(y_score_pairs_local, dtype=torch.float32, device=device)

    if distributed:
        y_true_all_t = gather_1d_tensor(y_true_t, world_size, rank)
        y_score_all_t = gather_1d_tensor(y_score_t, world_size, rank)
    else:
        y_true_all_t = y_true_t.cpu()
        y_score_all_t = y_score_t.cpu()

    if not is_main_process(rank):
        if distributed:
            dist.destroy_process_group()
        return

    # =========================
    # Финальные вычисления (rank 0)
    # =========================

    y_true_all = y_true_all_t.numpy()
    y_score_all = y_score_all_t.numpy()

    print(f"\nTotal examples: {total_examples}")
    print(f"Correct: {correct_total}, Wrong: {total_examples - correct_total}")
    acc_final = 100.0 * correct_total / total_examples if total_examples > 0 else 0.0
    print(f"Top-1 accuracy on A-OKVQA val: {acc_final:.2f}%")

    # latency
    lat_e2e_np = lat_e2e_all.numpy()
    lat_pre_np = lat_pre_all.numpy()
    lat_det_np = lat_det_all.numpy()

    lat_sorted = np.sort(lat_e2e_np)
    p50 = lat_sorted[int(0.5 * len(lat_sorted))]
    p95 = lat_sorted[int(0.95 * len(lat_sorted))]
    total_time = float(lat_e2e_np.sum())
    det_time = float(lat_det_np.sum())
    fps = total_examples / total_time if total_time > 0 else 0.0

    print(f"\nE2E latency P50: {p50*1000:.2f} ms")
    print(f"E2E latency P95: {p95*1000:.2f} ms")
    print(f"E2E latency avg: {(total_time/len(lat_e2e_np))*1000:.2f} ms")
    print(f"pre avg: {(lat_pre_np.mean())*1000:.2f} ms")
    print(f"det avg: {(lat_det_np.mean())*1000:.2f} ms")
    print(f"FPS: {fps:.2f}")

    # FLOPs / TFLOPs
    print(f"\nFLOPs per forward_mcq_logits: {flops_per_forward:.3e}")
    total_flops = flops_per_forward * num_forwards_total
    print(f"Total FLOPs (all forwards):   {total_flops:.3e}")
    tflops_effective = total_flops / det_time / 1e12 if det_time > 0 else 0.0
    print(f"Effective throughput:         {tflops_effective:.3f} TFLOPs")

    # pairwise метрики
    print("\nTotal pairs:", len(y_true_all))
    print("Positives (GT):", int(y_true_all.sum()))
    print("Negatives:", int((1 - y_true_all).sum()))

    threshold = 0.5
    y_pred_pair = (y_score_all >= threshold).astype(int)

    cm = confusion_matrix(y_true_all, y_pred_pair, labels=[1, 0])
    tp = cm[0, 0]
    fn = cm[0, 1]
    fp_ = cm[1, 0]
    tn = cm[1, 1]

    print("\nConfusion matrix (labels order: [1, 0])")
    print(cm)
    print(f"\nTP (GT pairs predicted positive): {tp}")
    print(f"FN (GT pairs predicted negative): {fn}")
    print(f"FP (neg pairs predicted positive): {fp_}")
    print(f"TN (neg pairs predicted negative): {tn}")

        # confusion matrix
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Positive (GT pair)", "Negative"],
    )
    fig, ax = plt.subplots(figsize=(4, 4))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    plt.title(f"nanoVLM MCQ pairwise confusion (threshold={threshold})")
    plt.tight_layout()
    fig.savefig(metrics_dir / "confusion_matrix.png", dpi=150)
    plt.close(fig)

    # ROC
    fpr, tpr, _ = roc_curve(y_true_all, y_score_all)
    roc_auc = auc(fpr, tpr)

    fig = plt.figure(figsize=(4, 4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("nanoVLM MCQ ROC (pairwise, A-OKVQA)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    fig.savefig(metrics_dir / "roc_curve.png", dpi=150)
    plt.close(fig)

        # PR
    precision, recall, _ = precision_recall_curve(y_true_all, y_score_all)
    ap = average_precision_score(y_true_all, y_score_all)

    fig = plt.figure(figsize=(4, 4))
    plt.plot(recall, precision, label=f"AP = {ap:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("nanoVLM MCQ Precision–Recall (pairwise, A-OKVQA)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    fig.savefig(metrics_dir / "pr_curve.png", dpi=150)
    plt.close(fig)

    # metrics vs threshold
    thresholds = np.linspace(0.0, 1.0, 51)
    accs, precs, recs, f1s = [], [], [], []

    for thr in thresholds:
        y_pred_thr = (y_score_all >= thr).astype(int)
        accs.append((y_pred_thr == y_true_all).mean())
        precs.append(precision_score(y_true_all, y_pred_thr, zero_division=0))
        recs.append(recall_score(y_true_all, y_pred_thr, zero_division=0))
        f1s.append(f1_score(y_true_all, y_pred_thr, zero_division=0))

    fig = plt.figure(figsize=(6, 4))
    plt.plot(thresholds, accs, label="Accuracy")
    plt.plot(thresholds, precs, label="Precision")
    plt.plot(thresholds, recs, label="Recall")
    plt.plot(thresholds, f1s, label="F1")
    plt.xlabel("Threshold")
    plt.ylabel("Score")
    plt.title("nanoVLM MCQ (A-OKVQA): metrics vs threshold")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    fig.savefig(metrics_dir / "metrics_vs_threshold.png", dpi=150)
    plt.close(fig)

    # распределение скорингов
    pos_scores = y_score_all[y_true_all == 1]
    neg_scores = y_score_all[y_true_all == 0]

    fig = plt.figure(figsize=(6, 4))
    plt.hist(pos_scores, bins=50, alpha=0.5, label="Positives", density=True)
    plt.hist(neg_scores, bins=50, alpha=0.5, label="Negatives", density=True)
    plt.xlabel("Score (softmax prob of caption among 4 choices)")
    plt.ylabel("Density")
    plt.title("nanoVLM MCQ (A-OKVQA): score distribution")
    plt.legend()
    plt.tight_layout()
    fig.savefig(metrics_dir / "score_distribution.png", dpi=150)
    plt.close(fig)

    # bias по ошибкам: распределение предсказанных вариантов среди ошибочных
    total_errors = sum(err_counts)
    print("\nBias among wrong predictions:")
    print("Total errors:", total_errors)
    for i, c in enumerate(err_counts):
        letter = LETTERS[i]
        frac = c / total_errors if total_errors > 0 else 0.0
        print(f"  {letter}: {c}  ({frac*100:.2f}%)")

    # ZIP со всеми картинками
    zip_path = metrics_dir.with_suffix(".zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in metrics_dir.glob("*.png"):
            z.write(p, arcname=p.name)
    print(f"\nSaved plots to {metrics_dir} and zip archive to {zip_path}")

    # корректно закрываем tqdm
    if is_main_process(rank) and isinstance(it, tqdm):
        it.close()

    # барьер для синхронизации и корректного выхода
    if distributed:
        dist.barrier()
        dist.destroy_process_group()
    print("Inference finished successfully.")



# =========================
# CLI
# =========================

def cli():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_id", type=str, default="lusxvr/nanoVLM",
                    help="Pretrained HF model id or base local folder")
    ap.add_argument("--metrics_dir", type=str, default="metrics_aokvqa",
                    help="Where to save plots and metrics")
    ap.add_argument("--ckpt_path", type=str, default="",
                    help="Optional path to checkpoint with model_state_dict")
    ap.add_argument("--max_examples", type=int, default=-1,
                    help="Limit number of validation examples (-1 = all)")
    ap.add_argument("--n_show_first", type=int, default=10,
                    help="How many examples to print (rank 0 only)")
    ap.add_argument("--seed", type=int, default=1234)
    args = ap.parse_args()
    run_inference(args)

if __name__ == "__main__":
    cli()


## Launch Inference

In [ ]:
%cd /kaggle/working/nanoVLM

!torchrun --nproc_per_node=2 infer_aokvqa_mcq_ddp.py \
    --model_id lusxvr/nanoVLM \
    --ckpt_path checkpoints_mcq_t4x2/mcq_finetuned/pytorch_model.bin \
    --max_examples 1000 \
    --n_show_first 10



# Tokens

In [ ]:
%%writefile /kaggle/working/nanoVLM/preprocess_aokvqa_mcq_tokens.py
import argparse
from pathlib import Path
from typing import List

import torch
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

from models.vision_language_model import VisionLanguageModel
from data.processors import get_tokenizer, get_image_processor

LETTERS = "ABCD"

def build_mc_prompt(question: str, choices: List[str]) -> str:
    # тот же промпт, что в DDP-тренере
    return (
        "You are a visual question-answering assistant.\n"
        "Read the question and choose exactly one correct option.\n\n"
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n"
        "Answer (A, B, C, or D):"
    )

def preprocess_split(model_id: str, out_dir: Path, split: str):
    print(f"Preprocessing split: {split}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Loading model:", model_id)
    vlm = VisionLanguageModel.from_pretrained(model_id).to(device).eval()

    cfg = vlm.cfg
    tokenizer = get_tokenizer(
        cfg.lm_tokenizer,
        cfg.vlm_extra_tokens,
        cfg.lm_chat_template,
    )
    imgproc = get_image_processor(cfg.max_img_size, cfg.vit_img_size)
    mp_image_token_length = cfg.mp_image_token_length

    ds = load_dataset("HuggingFaceM4/A-OKVQA", split=split)
    print(f"{split} size:", len(ds))

    all_input_ids = []
    all_gt_idx = []
    all_qid = []

    pbar = tqdm(range(len(ds)), desc=f"Preprocess {split}", dynamic_ncols=True)
    for idx in pbar:
        ex = ds[idx]
        img: Image.Image = ex["image"].convert("RGB")
        q = ex["question"]
        choices = list(ex["choices"])
        gt_idx = int(ex["correct_choice_idx"])
        qid = str(ex["question_id"])

        # один раз прогоняем imgproc, чтобы узнать n_imgs и корректно посчитать image_tokens
        proc_img, _ = imgproc(img)
        n_imgs = proc_img.shape[0]
        num_img_tokens = mp_image_token_length * n_imgs
        image_tokens = tokenizer.image_token * num_img_tokens

        prompt = build_mc_prompt(q, choices)
        messages = [{"role": "user", "content": image_tokens + prompt}]
        ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
        )
        all_input_ids.append(torch.tensor(ids, dtype=torch.long))
        all_gt_idx.append(gt_idx)
        all_qid.append(qid)

    out = {
        "input_ids": all_input_ids,
        "gt_idx": torch.tensor(all_gt_idx, dtype=torch.long),
        "qid": all_qid,
    }
    out_path = out_dir / f"aokvqa_mcq_tokens_{split}.pt"
    torch.save(out, out_path)
    print(f"Saved pretokenized {split} to: {out_path}")

def cli():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_id", type=str, default="lusxvr/nanoVLM")
    ap.add_argument("--out_dir", type=str, default="aokvqa_pretok")
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for split in ["train", "validation"]:
        preprocess_split(args.model_id, out_dir, split)

if __name__ == "__main__":
    cli()


In [ ]:
!python /kaggle/working/nanoVLM/preprocess_aokvqa_mcq_tokens.py \
  --model_id lusxvr/nanoVLM \
  --out_dir aokvqa_pretok


In [ ]:
import os
import json
import subprocess
from pathlib import Path

# ===========================
# Конфигурация
# ===========================
username = os.environ["KAGGLE_USERNAME"]     # твой Kaggle username
dataset_name = "aokvqa-pretok"   # НОВОЕ имя датасета для каждой загрузки

model_dir = "/kaggle/working/nanoVLM/aokvqa_pretok"
publish_dir = f"/kaggle/working/publish_{dataset_name}"

# ===========================
# Подготовка директории
# ===========================
os.makedirs(publish_dir, exist_ok=True)

# копируем файлы модели
os.system(f"cp -r {model_dir}/* {publish_dir}/")

# ===========================
# Создаём dataset-metadata.json
# ===========================
metadata = {
    "title": f"nanoVLM MCQ Model {dataset_name}",
    "id": f"{username}/{dataset_name}",
    "licenses": [{"name": "Apache-2.0"}]
}

with open(Path(publish_dir) / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# ===========================
# Публикация ДАТАСЕТА (НОВОГО)
# ===========================
print("Создаём новый датасет…")

proc = subprocess.run(
    ["kaggle", "datasets", "create", "-p", publish_dir, "--dir-mode", "zip"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print(proc.stdout)


# By Preprocessed on training

In [ ]:
%%writefile /kaggle/working/nanoVLM/train_aokvqa_mcq_ddp.py
import argparse, math, random, os, contextlib
from pathlib import Path
from typing import List, Dict, Any

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

from torch.cuda.amp import autocast, GradScaler

from models.vision_language_model import VisionLanguageModel
from data.processors import get_tokenizer, get_image_processor

# -----------------------------
# Глобальные настройки CUDA
# -----------------------------
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# -----------------------------
# MCQ head + forward_mcq_logits
# -----------------------------

class MCQHead(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 4),
        )

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        last = hidden[:, -1, :]
        return self.net(last)

def forward_mcq_logits(self,
                       input_ids: torch.Tensor,
                       images: torch.Tensor,
                       attention_mask: torch.Tensor | None = None) -> torch.Tensor:
    token_embd = self.decoder.token_embedding(input_ids)

    images_tensor = self._process_images(images, input_ids.device)
    if images_tensor is not None:
        image_embd = self.vision_encoder(images_tensor)
        image_embd = self.MP(image_embd)
        token_embd = self._replace_img_tokens_with_embd(input_ids, token_embd, image_embd)

    hidden, _ = self.decoder(
        token_embd,
        attention_mask=attention_mask,
        kv_cache=None,
        start_pos=0
    )
    logits = self.mcq_head(hidden)
    return logits

def mcq_forward(model,
                input_ids: torch.Tensor,
                images: torch.Tensor,
                distributed: bool,
                attention_mask: torch.Tensor | None = None) -> torch.Tensor:
    if distributed and isinstance(model, nn.parallel.DistributedDataParallel):
        return model.module.forward_mcq_logits(input_ids, images, attention_mask=attention_mask)
    else:
        return model.forward_mcq_logits(input_ids, images, attention_mask=attention_mask)

VisionLanguageModel.forward_mcq_logits = forward_mcq_logits

LETTER_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3}
IDX_TO_LETTER = "ABCD"

# -----------------------------
# Utils
# -----------------------------

def seed_everything(seed=1234):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def init_distributed():
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        rank = int(os.environ["RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        local_rank = int(os.environ.get("LOCAL_RANK", 0))
        dist.init_process_group(backend="nccl")
        torch.cuda.set_device(local_rank)
        return True, rank, world_size, local_rank
    else:
        return False, 0, 1, 0

def is_main_process(rank: int) -> bool:
    return rank == 0

# -----------------------------
# Датасет на основе .pt + HF изображений
# -----------------------------
class AOKVQAMCQTokensDataset(Dataset):
    """
    Ожидаем формат .pt как в preprocess_aokvqa_mcq_tokens.py:

        {
            "input_ids": list[ LongTensor (T_i) ],
            "gt_idx":    LongTensor [N],
            "qid":       list[str]
        }

    Для каждой строки:
      - берём input_ids[i] (1D LongTensor любой длины),
      - attention_mask = ones_like(input_ids),
      - label = gt_idx[i],
      - картинку берём из HuggingFaceM4/A-OKVQA по тому же индексу.
    """
    def __init__(self,
                 tokens_path: str,
                 split: str,
                 imgproc,
                 rank: int,
                 load_hf_images: bool = True):
        obj = torch.load(tokens_path, map_location="cpu")

        if not isinstance(obj, dict):
            raise ValueError(
                f"{tokens_path}: ожидался dict с ключами 'input_ids'/'gt_idx', а не {type(obj)}"
            )

        if "input_ids" not in obj or "gt_idx" not in obj:
            raise ValueError(
                f"{tokens_path} должен содержать ключи 'input_ids' (list[tensor]) и 'gt_idx' (tensor)"
            )

        input_ids_list = obj["input_ids"]
        gt_idx = obj["gt_idx"]

        if not isinstance(input_ids_list, list):
            raise ValueError(
                f"{tokens_path}['input_ids'] должен быть list, а не {type(input_ids_list)}"
            )
        if not isinstance(gt_idx, torch.Tensor) or gt_idx.ndim != 1:
            raise ValueError(
                f"{tokens_path}['gt_idx'] должен быть 1D LongTensor, а не {type(gt_idx)} с ndim={getattr(gt_idx, 'ndim', None)}"
            )
        if len(input_ids_list) != gt_idx.size(0):
            raise ValueError(
                f"{tokens_path}: len(input_ids)={len(input_ids_list)} != len(gt_idx)={gt_idx.size(0)}"
            )

        self.input_ids_list = input_ids_list
        self.labels = gt_idx
        self.qids = obj.get("qid", None)
        self.n = len(self.input_ids_list)

        self.imgproc = imgproc
        self.load_hf_images = load_hf_images
        if load_hf_images:
            self.hf_ds = load_dataset("HuggingFaceM4/A-OKVQA", split=split)

        if rank == 0:
            print(f"[tokens] {tokens_path}")
            print(f"         type={type(obj)}, N={self.n}")
            print(f"         input_ids[0].shape = {self.input_ids_list[0].shape}")
            print(f"         gt_idx.shape = {self.labels.shape}")
            if self.qids is not None:
                print(f"         qid[0] = {self.qids[0]}")

    def __len__(self):
        return self.n

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ids = self.input_ids_list[idx]
        ids = torch.as_tensor(ids, dtype=torch.long)

        attn = torch.ones_like(ids, dtype=torch.long)

        label = int(self.labels[idx].item())

        sample = {
            "input_ids": ids,
            "attention_mask": attn,
            "label": label,
        }

        if self.load_hf_images:
            hf_ex = self.hf_ds[int(idx)]
            img: Image.Image = hf_ex["image"].convert("RGB")
            proc_img, _ = self.imgproc(img)     # [n_imgs, 3, H, W]
            sample["images"] = proc_img

        return sample


# -----------------------------
# Настройка trainable модулей
# -----------------------------

def configure_trainable_modules(model, args, rank):
    if args.freeze_vision and hasattr(model, "vision_encoder"):
        for p in model.vision_encoder.parameters():
            p.requires_grad = False
        if is_main_process(rank):
            print("[freeze] vision_encoder")

    if args.freeze_proj and hasattr(model, "MP"):
        for p in model.MP.parameters():
            p.requires_grad = False
        if is_main_process(rank):
            print("[freeze] modality projector (MP)")
    else:
        if is_main_process(rank):
            print("[unfreeze] modality projector (MP)")

    dec = model.decoder

    if hasattr(dec, "layers"):
        layers = dec.layers
    elif hasattr(dec, "model") and hasattr(dec.model, "layers"):
        layers = dec.model.layers
    else:
        if is_main_process(rank):
            print("[warn] cannot find decoder layers (no .layers or .model.layers). "
                  "Decoder will stay as is.")
        return

    num_layers = len(layers)
    k = max(0, int(args.unfreeze_decoder_layers))

    if k == 0:
        for layer in layers:
            for p in layer.parameters():
                p.requires_grad = False
        if is_main_process(rank):
            print(f"[freeze] all {num_layers} decoder layers")
    else:
        cutoff = num_layers - k
        for i, layer in enumerate(layers):
            train_this = (i >= cutoff)
            for p in layer.parameters():
                p.requires_grad = train_this
        if is_main_process(rank):
            print(f"[freeze] first {cutoff} decoder layers, "
                  f"[unfreeze] last {k} decoder layers")

# -----------------------------
# TRAIN с DDP + AMP
# -----------------------------

def train(args):
    seed_everything(args.seed)

    distributed, rank, world_size, local_rank = init_distributed()
    device = torch.device("cuda", local_rank) if torch.cuda.is_available() else torch.device("cpu")

    use_amp = not getattr(args, "no_amp", False)

    if is_main_process(rank):
        print(f"Distributed: {distributed} | world_size={world_size} | rank={rank}")
        print(f"Device: {device}")
        print(f"Loading model: {args.model_id}")
        print(f"use_amp = {use_amp}")

    model = VisionLanguageModel.from_pretrained(args.model_id).to(device)

    hidden_dim = model.decoder.token_embedding.embedding_dim
    model.mcq_head = MCQHead(hidden_dim, dropout=args.head_dropout).to(device)

    max_img_size = model.cfg.max_img_size
    vit_img_size = model.cfg.vit_img_size

    configure_trainable_modules(model, args, rank)

    if distributed:
        model = nn.parallel.DistributedDataParallel(
            model,
            device_ids=[local_rank],
            output_device=local_rank,
            find_unused_parameters=False,
        )

    cfg = model.module.cfg if distributed else model.cfg
    tokenizer = get_tokenizer(
        cfg.lm_tokenizer,
        cfg.vlm_extra_tokens,
        cfg.lm_chat_template,
    )
    imgproc = get_image_processor(max_img_size, vit_img_size)

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id

    # Датасеты из .pt + HF-изображения
    train_dataset = AOKVQAMCQTokensDataset(
        tokens_path=args.train_tokens_path,
        split="train",
        imgproc=imgproc,
        rank=rank,
        load_hf_images=True,
    )
    val_dataset = AOKVQAMCQTokensDataset(
        tokens_path=args.val_tokens_path,
        split="validation",
        imgproc=imgproc,
        rank=rank,
        load_hf_images=True,
    )

    if is_main_process(rank):
        print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

    if distributed:
        train_sampler = DistributedSampler(
            train_dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=True,
        )
    else:
        train_sampler = None

    def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        bsz = len(batch)
        max_len = max(ex["input_ids"].size(0) for ex in batch)

        input_ids = torch.full((bsz, max_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((bsz, max_len), dtype=torch.long)

        images_list = []
        labels = torch.empty(bsz, dtype=torch.long)

        for i, ex in enumerate(batch):
            ids = ex["input_ids"]
            attn = ex["attention_mask"]
            L = ids.size(0)
            input_ids[i, :L] = ids
            attention_mask[i, :L] = attn[:L]

            img = ex["images"]  # [n_imgs_i, 3, H, W]
            images_list.append(img)

            labels[i] = int(ex["label"])

        images = torch.cat(images_list, dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "images": images,
            "labels": labels,
        }

    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        sampler=train_sampler,
        shuffle=(train_sampler is None),
        num_workers=args.num_workers,
        pin_memory=True,
        collate_fn=collate_fn,
        drop_last=False,
        persistent_workers=(args.num_workers > 0),
        prefetch_factor=2 if args.num_workers > 0 else None,
    )

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if is_main_process(rank):
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params_cnt = sum(p.numel() for p in trainable_params)
        print(f"Total params: {total_params/1e6:.2f}M | Trainable: {trainable_params_cnt/1e6:.2f}M")

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=args.lr,
        betas=(0.9, 0.95),
        weight_decay=args.weight_decay,
    )
    scaler = GradScaler(enabled=use_amp)

    updates_per_epoch = math.ceil(len(train_loader) / max(1, args.grad_accum))
    total_steps = updates_per_epoch * args.epochs
    warmup_steps = int(args.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, (total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
    global_step, best_val = 0, -1.0

    @torch.no_grad()
    def evaluate_mcq(eval_ds: Dataset) -> float:
        assert is_main_process(rank)
        model.eval()
        correct = 0
        total = len(eval_ds)
        pbar = tqdm(range(total), desc="Eval-MCQ", dynamic_ncols=True)
        for idx in pbar:
            ex = eval_ds[idx]
            ids = ex["input_ids"].unsqueeze(0).to(device)
            attn = ex["attention_mask"].unsqueeze(0).to(device)
            imgs = ex["images"].to(device)
            label = int(ex["label"])

            with autocast(enabled=use_amp):
                logits = mcq_forward(
                    model,
                    ids,
                    imgs,
                    distributed,
                    attention_mask=attn,
                )
            pred_idx = int(logits.argmax(dim=-1).item())
            correct += int(pred_idx == label)

        model.train()
        return 100.0 * correct / total

    optimizer.zero_grad(set_to_none=True)

    for epoch in range(args.epochs):
        if distributed and isinstance(train_sampler, DistributedSampler):
            train_sampler.set_epoch(epoch)

        if is_main_process(rank):
            epoch_pbar = tqdm(
                enumerate(train_loader),
                total=len(train_loader),
                desc=f"Epoch {epoch+1}/{args.epochs}",
                dynamic_ncols=True,
            )
        else:
            epoch_pbar = enumerate(train_loader)

        for step_in_epoch, batch in epoch_pbar:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            images = batch["images"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            micro_step = step_in_epoch % args.grad_accum
            sync_grad = (micro_step == args.grad_accum - 1)

            if distributed and isinstance(model, nn.parallel.DistributedDataParallel) and not sync_grad:
                ctx = model.no_sync()
            else:
                ctx = contextlib.nullcontext()

            with ctx:
                with autocast(enabled=use_amp):
                    logits = mcq_forward(
                        model,
                        input_ids,
                        images,
                        distributed,
                        attention_mask=attention_mask,
                    )
                    loss = loss_fn(logits, labels)
                loss = loss / args.grad_accum
                scaler.scale(loss).backward()

            if sync_grad:
                if use_amp:
                    scaler.unscale_(optimizer)

                clip_grad_norm_(trainable_params, args.grad_clip)

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if is_main_process(rank):
                    epoch_pbar.set_postfix(
                        loss=f"{loss.item()*args.grad_accum:.4f}",
                        lr=f"{scheduler.get_last_lr()[0]:.2e}",
                    )

                if is_main_process(rank) and (global_step % args.eval_every == 0):
                    val_acc = evaluate_mcq(val_dataset)
                    print(f"\n[Eval] step={global_step} | val MCQ acc = {val_acc:.2f}%")
                    if val_acc > best_val:
                        best_val = val_acc
                        ckpt_path = Path(args.output_dir)
                        ckpt_path.mkdir(parents=True, exist_ok=True)
                        ckpt_path = ckpt_path / f"best_mcq_{val_acc:.2f}.pt"
                        print(f"[Checkpoint] Saving to {ckpt_path}")
                        torch.save({
                            "model_state_dict": model.module.state_dict() if distributed else model.state_dict(),
                            "val_acc": val_acc,
                            "step": global_step
                        }, ckpt_path)

    if is_main_process(rank):
        final_acc = evaluate_mcq(val_dataset)
        print(f"\n[Final] val MCQ acc = {final_acc:.2f}%  (best={best_val:.2f}%)")

        full_out = Path(args.output_dir) / "mcq_finetuned"
        full_out.mkdir(parents=True, exist_ok=True)
        state_dict = model.module.state_dict() if distributed else model.state_dict()
        torch.save(state_dict, full_out / "pytorch_model.bin")
        print(f"Saved state_dict to: {full_out}")

    if distributed:
        dist.destroy_process_group()

def cli():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_id", default="lusxvr/nanoVLM")
    ap.add_argument("--output_dir", default="checkpoints_mcq")
    ap.add_argument("--epochs", type=int, default=1)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--weight_decay", type=float, default=0.01)
    ap.add_argument("--warmup_ratio", type=float, default=0.10)

    ap.add_argument("--batch_size", type=int, default=2)
    ap.add_argument("--grad_accum", type=int, default=4)
    ap.add_argument("--grad_clip", type=float, default=1.0)
    ap.add_argument("--eval_every", type=int, default=500)

    ap.add_argument("--num_workers", type=int, default=4)

    ap.add_argument("--label_smoothing", type=float, default=0.1)
    ap.add_argument("--head_dropout", type=float, default=0.1)
    ap.add_argument("--unfreeze_decoder_layers", type=int, default=4)

    ap.add_argument("--freeze_vision", action="store_true", default=True)
    ap.add_argument("--freeze_proj", action="store_true", default=False)
    ap.add_argument("--seed", type=int, default=1234)

    ap.add_argument("--train_tokens_path",
                    type=str,
                    default="/kaggle/input/aokvqa-pretok/aokvqa_mcq_tokens_train.pt")
    ap.add_argument("--val_tokens_path",
                    type=str,
                    default="/kaggle/input/aokvqa-pretok/aokvqa_mcq_tokens_validation.pt")

    ap.add_argument("--no_amp", action="store_true",
                    help="Отключить mixed precision (по умолчанию включён)")

    args = ap.parse_args()
    train(args)

if __name__ == "__main__":
    cli()


In [ ]:
%cd /kaggle/working/nanoVLM

!torchrun --nproc_per_node=2 train_aokvqa_mcq_ddp.py \
  --train_tokens_path /kaggle/input/aokvqa-pretok/aokvqa_mcq_tokens_train.pt \
  --val_tokens_path /kaggle/input/aokvqa-pretok/aokvqa_mcq_tokens_validation.pt


## Launch

In [ ]:
%%writefile /kaggle/working/nanoVLM/infer_aokvqa_mcq_ddp.py
import argparse, os, math, time, random
from pathlib import Path
from typing import List, Dict, Any

import torch
import torch.nn as nn
import torch.distributed as dist
import zipfile
import matplotlib
matplotlib.use("Agg")  # рисуем в память, без GUI

import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, load_dataset_builder
from PIL import Image
from tqdm import tqdm

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score,
)

from thop import profile

from models.vision_language_model import VisionLanguageModel
from data.processors import get_tokenizer, get_image_processor

# =========================
# MCQ head + forward_mcq_logits
# (та же, что в train_aokvqa_mcq_ddp.py)
# =========================

class MCQHead(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 4),
        )

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        last = hidden[:, -1, :]
        return self.net(last)

def forward_mcq_logits(self, input_ids: torch.Tensor, images: torch.Tensor, attention_mask=None) -> torch.Tensor:
    token_embd = self.decoder.token_embedding(input_ids)

    images_tensor = self._process_images(images, input_ids.device)
    if images_tensor is not None:
        image_embd = self.vision_encoder(images_tensor)
        image_embd = self.MP(image_embd)
        token_embd = self._replace_img_tokens_with_embd(input_ids, token_embd, image_embd)

    hidden, _ = self.decoder(
        token_embd,
        attention_mask=attention_mask,
        kv_cache=None,
        start_pos=0
    )
    logits = self.mcq_head(hidden)  # [B, 4]
    return logits

# патчим класс той же функцией, что в трейне
VisionLanguageModel.forward_mcq_logits = forward_mcq_logits

LETTERS = "ABCD"

# =========================
# DDP utils
# =========================

def seed_everything(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def init_distributed():
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        rank = int(os.environ["RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        local_rank = int(os.environ.get("LOCAL_RANK", 0))
        dist.init_process_group(backend="nccl")
        torch.cuda.set_device(local_rank)
        return True, rank, world_size, local_rank
    else:
        return False, 0, 1, 0

def is_main_process(rank: int) -> bool:
    return rank == 0

def mcq_forward(model, input_ids, images, distributed: bool, attention_mask=None):
    if distributed and isinstance(model, nn.parallel.DistributedDataParallel):
        return model.module.forward_mcq_logits(input_ids, images, attention_mask=attention_mask)
    else:
        return model.forward_mcq_logits(input_ids, images, attention_mask=attention_mask)

# =========================
# Dataset
# =========================

class AOKVQAMCQDataset(torch.utils.data.Dataset):
    def __init__(self, split="validation"):
        self.ds = load_dataset("HuggingFaceM4/A-OKVQA", split=split)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.ds[int(idx)]
        return {
            "image": ex["image"],
            "question": ex["question"],
            "choices": list(ex["choices"]),
            "gt_idx": int(ex["correct_choice_idx"]),
            "qid": str(ex["question_id"]),
        }

def build_mc_prompt(question: str, choices: List[str]) -> str:
    return (
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n"
        "Answer:"
    )

# =========================
# All-gather helper for 1D tensors (variable length)
# =========================

def gather_1d_tensor(t: torch.Tensor, world_size: int, rank: int):
    """
    t: 1D tensor on each rank (can be different length).
    Возвращает на rank 0 конкатенированный tensor, на остальных None.
    """
    device = t.device
    length = torch.tensor([t.numel()], device=device, dtype=torch.long)
    lengths = [torch.zeros_like(length) for _ in range(world_size)]
    dist.all_gather(lengths, length)
    lengths = [l.item() for l in lengths]
    max_len = max(lengths)

    if t.numel() < max_len:
        pad = torch.zeros(max_len - t.numel(), dtype=t.dtype, device=device)
        t_padded = torch.cat([t, pad], dim=0)
    else:
        t_padded = t

    gather_list = [torch.zeros_like(t_padded) for _ in range(world_size)]
    dist.all_gather(gather_list, t_padded)

    if rank == 0:
        parts = []
        for i, g in enumerate(gather_list):
            parts.append(g[:lengths[i]].cpu())
        return torch.cat(parts, dim=0)
    else:
        return None

# =========================
# Main inference routine
# =========================
def run_inference(args):
    seed_everything(args.seed)

    # создаём папку под метрики до инициализации DDP (просто rank=0 по умолчанию)
    metrics_dir = Path(args.metrics_dir)
    metrics_dir.mkdir(parents=True, exist_ok=True)

    distributed, rank, world_size, local_rank = init_distributed()
    device = torch.device("cuda", local_rank) if torch.cuda.is_available() else torch.device("cpu")
    metrics_dir = Path(args.metrics_dir)

    if is_main_process(rank):
        metrics_dir.mkdir(parents=True, exist_ok=True)
        print(f"Distributed: {distributed} | world_size={world_size} | rank={rank}")
        print("Device:", device)
        print("Loading model:", args.model_id)

    # базовая модель
    model = VisionLanguageModel.from_pretrained(args.model_id).to(device)

    # навешиваем MCQ-голову с той же архитектурой
    hidden_dim = model.decoder.token_embedding.embedding_dim
    model.mcq_head = MCQHead(hidden_dim, dropout=0.1).to(device)

    # если есть чекпоинт — подгружаем веса
    if args.ckpt_path is not None and args.ckpt_path != "":
        if is_main_process(rank):
            print("Loading checkpoint:", args.ckpt_path)
        ckpt = torch.load(args.ckpt_path, map_location="cpu")
        state_dict = ckpt.get("model_state_dict", ckpt)
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        if is_main_process(rank):
            print("Loaded state_dict.")
            if missing:
                print("  Missing keys:", missing)
            if unexpected:
                print("  Unexpected keys:", unexpected)

    if distributed:
        model = nn.parallel.DistributedDataParallel(
            model,
            device_ids=[local_rank],
            output_device=local_rank,
            find_unused_parameters=False,
        )

    cfg = model.module.cfg if distributed else model.cfg
    tokenizer = get_tokenizer(cfg.lm_tokenizer, cfg.vlm_extra_tokens, cfg.lm_chat_template)
    imgproc = get_image_processor(cfg.max_img_size, cfg.vit_img_size)
    mp_image_token_length = cfg.mp_image_token_length

    # инфо о датасете
    if is_main_process(rank):
        builder = load_dataset_builder("HuggingFaceM4/A-OKVQA")
        info = builder.info
        download_gb = info.download_size / (1024**3)
        dataset_gb  = info.dataset_size  / (1024**3)
        print(f"Плановый объём скачивания (raw): {download_gb:.2f} GB")
        print(f"Размер закэшированного датасета: {dataset_gb:.2f} GB")

    ds_val = AOKVQAMCQDataset(split="validation")
    total_ds = len(ds_val)

    if args.max_examples > 0:
        total_ds = min(total_ds, args.max_examples)

    if is_main_process(rank):
        print("A-OKVQA val size:", len(ds_val))
        print("Using first", total_ds, "examples for eval")

    # FLOPs на одном примере (только rank 0)
    flops_per_forward = None
    params = None
    if is_main_process(rank):
        class MCQWrapper(nn.Module):
            def __init__(self, vlm):
                super().__init__()
                self.vlm = vlm

            def forward(self, input_ids, images):
                return self.vlm.forward_mcq_logits(input_ids, images)

        mcq_wrapper = MCQWrapper(model.module if distributed else model).to(device).eval()

        with torch.no_grad():
            ex0 = ds_val[0]
            img0: Image.Image = ex0["image"].convert("RGB")
            q0 = ex0["question"]
            choices0 = list(ex0["choices"])

            proc_img0, _ = imgproc(img0)
            n_imgs0 = proc_img0.shape[0]
            num_img_tokens0 = mp_image_token_length * n_imgs0
            image_tokens0 = tokenizer.image_token * num_img_tokens0

            prompt0 = build_mc_prompt(q0, choices0)
            messages0 = [{"role": "user", "content": image_tokens0 + prompt0}]
            ids0 = tokenizer.apply_chat_template(
                messages0,
                tokenize=True,
                add_generation_prompt=True
            )
            input_ids0 = torch.tensor(ids0, dtype=torch.long, device=device).unsqueeze(0)

            flops_per_forward, params = profile(
                mcq_wrapper,
                inputs=(input_ids0, proc_img0.to(device)),
                verbose=False
            )

        print(f"FLOPs per forward_mcq_logits: {flops_per_forward:.3e}")
        print(f"Params in MCQ pipeline:       {params:.3e}")

    # раскидываем flops/params на все процессы
    if distributed:
        flops_tensor = torch.zeros(2, dtype=torch.float64, device=device)
        if is_main_process(rank):
            flops_tensor[0] = float(flops_per_forward if flops_per_forward is not None else 0.0)
            flops_tensor[1] = float(params if params is not None else 0.0)
        dist.broadcast(flops_tensor, src=0)
        flops_per_forward = float(flops_tensor[0].item())
        params = float(flops_tensor[1].item())

    # раздаём индексы по rank'ам
    indices = list(range(total_ds))
    per_rank = (total_ds + world_size - 1) // world_size
    start = rank * per_rank
    end = min(start + per_rank, total_ds)
    local_indices = indices[start:end]

    if is_main_process(rank):
        print(f"indices per rank ~ {per_rank}, rank0 range = [0, {min(per_rank, total_ds)})")

    model.eval()

    correct_local = 0
    num_forwards_local = 0
    y_true_pairs_local = []
    y_score_pairs_local = []

    lat_e2e_local = []
    lat_pre_local = []
    lat_det_local = []

    err_pred_counts_local = [0, 0, 0, 0]

    try:
        from IPython.display import display
    except ImportError:
        display = None

    logged = 0
    n_show_first = args.n_show_first
    seen_local = 0

    if is_main_process(rank):
        it = tqdm(local_indices, desc="Inference (rank 0)", dynamic_ncols=True)
    else:
        it = local_indices

    for ex_idx in it:
        t0 = time.perf_counter()

        ex = ds_val[ex_idx]
        img: Image.Image = ex["image"].convert("RGB")
        q = ex["question"]
        choices = list(ex["choices"])
        gt_idx = int(ex["gt_idx"])

        # препроцессинг
        t_pre0 = time.perf_counter()
        proc_img, _ = imgproc(img)
        n_imgs = proc_img.shape[0]
        num_img_tokens = mp_image_token_length * n_imgs
        image_tokens = tokenizer.image_token * num_img_tokens

        prompt = build_mc_prompt(q, choices)
        messages = [{"role": "user", "content": image_tokens + prompt}]
        ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True
        )
        input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
        t_pre1 = time.perf_counter()

        # forward
        t_det0 = time.perf_counter()
        logits = mcq_forward(model, input_ids, proc_img.to(device), distributed).squeeze(0)
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        t_det1 = time.perf_counter()

        num_forwards_local += 1

        pred_idx = int(np.argmax(probs))
        is_correct = (pred_idx == gt_idx)
        seen_local += 1
        if is_correct:
            correct_local += 1
        else:
            err_pred_counts_local[pred_idx] += 1

        # pairwise статистика
        for j in range(len(choices)):
            y_true_pairs_local.append(1 if j == gt_idx else 0)
            y_score_pairs_local.append(float(probs[j]))

        # логирование только первых n_show_first примеров на rank 0
        if is_main_process(rank) and logged < n_show_first:
            logged += 1
            print(f"\n=== Example {logged} (global idx {ex_idx}) ===")
            if display is not None:
                display(img)
            print("Question:", q)
            print("Choices and probs:")
            for i, (p, ch) in enumerate(zip(probs, choices)):
                letter = LETTERS[i]
                marker_gt = "<-- GT" if i == gt_idx else ""
                marker_pred = "<-- PRED" if i == pred_idx else ""
                print(f"  {letter}) {p:.4f} | {ch} {marker_gt} {marker_pred}")
            print(f"Final predicted: {LETTERS[pred_idx]}  (correct={is_correct})")

        # latency
        t1 = time.perf_counter()
        lat_pre_local.append(t_pre1 - t_pre0)
        lat_det_local.append(t_det1 - t_det0)
        lat_e2e_local.append(t1 - t0)

        # обновление прогресса глобальными числами
        if distributed:
            stats = torch.tensor(
                [correct_local, seen_local],
                dtype=torch.long,
                device=device,
            )
            dist.all_reduce(stats, op=dist.ReduceOp.SUM)
            correct_now, seen_now = stats.tolist()
        else:
            correct_now, seen_now = correct_local, seen_local

        if is_main_process(rank) and isinstance(it, tqdm):
            wrong_now = seen_now - correct_now
            acc_now = 100.0 * correct_now / max(1, seen_now)
            it.set_postfix(
                correct=correct_now,
                wrong=wrong_now,
                acc=f"{acc_now:.2f}%"
            )

    # =========================
    # Сбор метрик с ранков
    # =========================

    local_counts = torch.tensor(
        [len(local_indices), correct_local, num_forwards_local],
        dtype=torch.long, device=device
    )

    if distributed:
        dist.reduce(local_counts, dst=0, op=dist.ReduceOp.SUM)

    if is_main_process(rank):
        total_examples, correct_total, num_forwards_total = local_counts.tolist()
    else:
        total_examples = correct_total = num_forwards_total = None

    # ошибки по вариантам
    err_counts_t = torch.tensor(err_pred_counts_local, dtype=torch.long, device=device)
    if distributed:
        dist.reduce(err_counts_t, dst=0, op=dist.ReduceOp.SUM)
    if is_main_process(rank):
        err_counts = err_counts_t.cpu().tolist()
    else:
        err_counts = None

    # времена
    lat_e2e_t = torch.tensor(lat_e2e_local, dtype=torch.float64, device=device)
    lat_pre_t = torch.tensor(lat_pre_local, dtype=torch.float64, device=device)
    lat_det_t = torch.tensor(lat_det_local, dtype=torch.float64, device=device)

    if distributed:
        lat_e2e_all = gather_1d_tensor(lat_e2e_t, world_size, rank)
        lat_pre_all = gather_1d_tensor(lat_pre_t, world_size, rank)
        lat_det_all = gather_1d_tensor(lat_det_t, world_size, rank)
    else:
        lat_e2e_all = lat_e2e_t.cpu()
        lat_pre_all = lat_pre_t.cpu()
        lat_det_all = lat_det_t.cpu()

    # pairwise y_true / y_score
    y_true_t = torch.tensor(y_true_pairs_local, dtype=torch.int64, device=device)
    y_score_t = torch.tensor(y_score_pairs_local, dtype=torch.float32, device=device)

    if distributed:
        y_true_all_t = gather_1d_tensor(y_true_t, world_size, rank)
        y_score_all_t = gather_1d_tensor(y_score_t, world_size, rank)
    else:
        y_true_all_t = y_true_t.cpu()
        y_score_all_t = y_score_t.cpu()

    if not is_main_process(rank):
        if distributed:
            dist.destroy_process_group()
        return

    # =========================
    # Финальные вычисления (rank 0)
    # =========================

    y_true_all = y_true_all_t.numpy()
    y_score_all = y_score_all_t.numpy()

    print(f"\nTotal examples: {total_examples}")
    print(f"Correct: {correct_total}, Wrong: {total_examples - correct_total}")
    acc_final = 100.0 * correct_total / total_examples if total_examples > 0 else 0.0
    print(f"Top-1 accuracy on A-OKVQA val: {acc_final:.2f}%")

    # latency
    lat_e2e_np = lat_e2e_all.numpy()
    lat_pre_np = lat_pre_all.numpy()
    lat_det_np = lat_det_all.numpy()

    lat_sorted = np.sort(lat_e2e_np)
    p50 = lat_sorted[int(0.5 * len(lat_sorted))]
    p95 = lat_sorted[int(0.95 * len(lat_sorted))]
    total_time = float(lat_e2e_np.sum())
    det_time = float(lat_det_np.sum())
    fps = total_examples / total_time if total_time > 0 else 0.0

    print(f"\nE2E latency P50: {p50*1000:.2f} ms")
    print(f"E2E latency P95: {p95*1000:.2f} ms")
    print(f"E2E latency avg: {(total_time/len(lat_e2e_np))*1000:.2f} ms")
    print(f"pre avg: {(lat_pre_np.mean())*1000:.2f} ms")
    print(f"det avg: {(lat_det_np.mean())*1000:.2f} ms")
    print(f"FPS: {fps:.2f}")

    # FLOPs / TFLOPs
    print(f"\nFLOPs per forward_mcq_logits: {flops_per_forward:.3e}")
    total_flops = flops_per_forward * num_forwards_total
    print(f"Total FLOPs (all forwards):   {total_flops:.3e}")
    tflops_effective = total_flops / det_time / 1e12 if det_time > 0 else 0.0
    print(f"Effective throughput:         {tflops_effective:.3f} TFLOPs")

    # pairwise метрики
    print("\nTotal pairs:", len(y_true_all))
    print("Positives (GT):", int(y_true_all.sum()))
    print("Negatives:", int((1 - y_true_all).sum()))

    threshold = 0.5
    y_pred_pair = (y_score_all >= threshold).astype(int)

    cm = confusion_matrix(y_true_all, y_pred_pair, labels=[1, 0])
    tp = cm[0, 0]
    fn = cm[0, 1]
    fp_ = cm[1, 0]
    tn = cm[1, 1]

    print("\nConfusion matrix (labels order: [1, 0])")
    print(cm)
    print(f"\nTP (GT pairs predicted positive): {tp}")
    print(f"FN (GT pairs predicted negative): {fn}")
    print(f"FP (neg pairs predicted positive): {fp_}")
    print(f"TN (neg pairs predicted negative): {tn}")

    # confusion matrix
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Positive (GT pair)", "Negative"],
    )
    fig, ax = plt.subplots(figsize=(4, 4))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    plt.title(f"nanoVLM MCQ pairwise confusion (threshold={threshold})")
    plt.tight_layout()
    fig.savefig(metrics_dir / "confusion_matrix.png", dpi=150)
    plt.close(fig)

    # ROC
    fpr, tpr, _ = roc_curve(y_true_all, y_score_all)
    roc_auc = auc(fpr, tpr)

    fig = plt.figure(figsize=(4, 4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("nanoVLM MCQ ROC (pairwise, A-OKVQA)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    fig.savefig(metrics_dir / "roc_curve.png", dpi=150)
    plt.close(fig)

    # PR
    precision, recall, _ = precision_recall_curve(y_true_all, y_score_all)
    ap = average_precision_score(y_true_all, y_score_all)

    fig = plt.figure(figsize=(4, 4))
    plt.plot(recall, precision, label=f"AP = {ap:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("nanoVLM MCQ Precision–Recall (pairwise, A-OKVQA)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    fig.savefig(metrics_dir / "pr_curve.png", dpi=150)
    plt.close(fig)

    # metrics vs threshold
    thresholds = np.linspace(0.0, 1.0, 51)
    accs, precs, recs, f1s = [], [], [], []

    for thr in thresholds:
        y_pred_thr = (y_score_all >= thr).astype(int)
        accs.append((y_pred_thr == y_true_all).mean())
        precs.append(precision_score(y_true_all, y_pred_thr, zero_division=0))
        recs.append(recall_score(y_true_all, y_pred_thr, zero_division=0))
        f1s.append(f1_score(y_true_all, y_pred_thr, zero_division=0))

    fig = plt.figure(figsize=(6, 4))
    plt.plot(thresholds, accs, label="Accuracy")
    plt.plot(thresholds, precs, label="Precision")
    plt.plot(thresholds, recs, label="Recall")
    plt.plot(thresholds, f1s, label="F1")
    plt.xlabel("Threshold")
    plt.ylabel("Score")
    plt.title("nanoVLM MCQ (A-OKVQA): metrics vs threshold")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    fig.savefig(metrics_dir / "metrics_vs_threshold.png", dpi=150)
    plt.close(fig)

    # распределение скорингов
    pos_scores = y_score_all[y_true_all == 1]
    neg_scores = y_score_all[y_true_all == 0]

    fig = plt.figure(figsize=(6, 4))
    plt.hist(pos_scores, bins=50, alpha=0.5, label="Positives", density=True)
    plt.hist(neg_scores, bins=50, alpha=0.5, label="Negatives", density=True)
    plt.xlabel("Score (softmax prob of caption among 4 choices)")
    plt.ylabel("Density")
    plt.title("nanoVLM MCQ (A-OKVQA): score distribution")
    plt.legend()
    plt.tight_layout()
    fig.savefig(metrics_dir / "score_distribution.png", dpi=150)
    plt.close(fig)

    # bias по ошибкам
    total_errors = sum(err_counts)
    print("\nBias among wrong predictions:")
    print("Total errors:", total_errors)
    for i, c in enumerate(err_counts):
        letter = LETTERS[i]
        frac = c / total_errors if total_errors > 0 else 0.0
        print(f"  {letter}: {c}  ({frac*100:.2f}%)")

    # ZIP со всеми картинками
    zip_path = metrics_dir.with_suffix(".zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in metrics_dir.glob("*.png"):
            z.write(p, arcname=p.name)
    print(f"\nSaved plots to {metrics_dir} and zip archive to {zip_path}")

    if is_main_process(rank) and isinstance(it, tqdm):
        it.close()

    if distributed:
        dist.barrier()
        dist.destroy_process_group()
    print("Inference finished successfully.")

# =========================
# CLI
# =========================

def cli():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_id", type=str, default="lusxvr/nanoVLM",
                    help="Pretrained HF model id or base local folder")
    ap.add_argument("--metrics_dir", type=str, default="metrics_aokvqa",
                    help="Where to save plots and metrics")
    ap.add_argument("--ckpt_path", type=str, default="",
                    help="Optional path to checkpoint with model_state_dict")
    ap.add_argument("--max_examples", type=int, default=-1,
                    help="Limit number of validation examples (-1 = all)")
    ap.add_argument("--n_show_first", type=int, default=10,
                    help="How many examples to print (rank 0 only)")
    ap.add_argument("--seed", type=int, default=1234)
    args = ap.parse_args()
    run_inference(args)

if __name__ == "__main__":
    cli()


In [ ]:
%cd /kaggle/working/nanoVLM

!torchrun --nproc_per_node=2 infer_aokvqa_mcq_ddp.py \
    --model_id lusxvr/nanoVLM \
    --ckpt_path /kaggle/working/nanoVLM/checkpoints_mcq/best_mcq_72.40.pt \
    --max_examples 1000 \
    --n_show_first 10